In [2]:
import random
from concurrent.futures import ThreadPoolExecutor

# Global constants
W_MAX = 50.0  # Maximum water capacity per cell (in liters)

# Plant parameters: (Ideal Moisture %, Water Factor K_plant)
PLANT_DB = {
    "succulent": (20.0, 0.15),
    "grass":     (50.0, 0.30),
    "flower":    (60.0, 0.35),
    "vegetable": (70.0, 0.45),
}

# --- Helper Functions Matching Pseudocode ---

def CalculateRequiredWater(plant_type, soil_moisture, temperature):
    ideal_moisture, k_plant = PLANT_DB.get(plant_type.lower(), (50.0, 0.30))
    evaporation_factor = 0.2
    w_req = (ideal_moisture - soil_moisture) * k_plant + (evaporation_factor * temperature)
    return max(0.0, round(w_req, 2))

def EvaluateFitness(schedule_val, w_req):
    w1, w2, w3 = 2.0, 2.5, 0.1  # Penalties for underwatering, wastage, and total usage
    penalty_under = w1 * max(0.0, w_req - schedule_val)
    penalty_waste = w2 * max(0.0, schedule_val - w_req)
    return 1.0 / (1.0 + penalty_under + penalty_waste + (w3 * schedule_val))

def GetMooreNeighbors(i, j, M, N, Schedule, Fitness):
    neighbors = []
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            ni, nj = i + di, j + dj
            if 0 <= ni < M and 0 <= nj < N:
                neighbors.append((Schedule[ni][nj], Fitness[ni][nj]))
    return neighbors

def ComputeSeepage(current_schedule, neighbors):
    beta = 0.05  # Soil diffusion coefficient
    return beta * sum(n_sched - current_schedule for n_sched, _ in neighbors)

def ArgMaxFitness(neighbors):
    if not neighbors:
        return 0.0
    best_neighbor = max(neighbors, key=lambda x: x[1])
    return best_neighbor[0]  # Returns Schedule of neighbor with highest Fitness

def UpdateRule(current_schedule, w_req, seepage, best_neighbor_sched):
    c1, c2 = 0.5, 0.2
    epsilon = random.uniform(-0.5, 0.5)  # Small exploration noise
    return (
        current_schedule
        + c1 * (w_req - current_schedule - seepage)
        + c2 * (best_neighbor_sched - current_schedule)
        + epsilon
    )

def Clamp(value, min_val, max_val):
    return max(min_val, min(value, max_val))

def TotalWaterUsed(Schedule):
    return round(sum(sum(row) for row in Schedule), 2)

# --- Main Algorithm ---

def main():
    # INPUT M, N, Temperature, Max_Iter
    M = int(input("Enter number of rows (M): "))
    N = int(input("Enter number of columns (N): "))
    Temperature = float(input("Enter ambient temperature (°C): "))
    Max_Iter = int(input("Enter max iterations (Max_Iter): "))

    Plant_Type = [["" for _ in range(N)] for _ in range(M)]
    Soil_Moisture = [[0.0 for _ in range(N)] for _ in range(M)]
    W_req = [[0.0 for _ in range(N)] for _ in range(M)]
    Schedule = [[0.0 for _ in range(N)] for _ in range(M)]
    Fitness = [[0.0 for _ in range(N)] for _ in range(M)]

    print("\nAvailable plant types: succulent, grass, flower, vegetable")

    # FOR each cell (i, j) in M x N DO
    for i in range(M):
        for j in range(N):
            print(f"\n--- Cell ({i}, {j}) ---")
            Plant_Type[i][j] = input("  Enter Plant Type: ").strip()
            Soil_Moisture[i][j] = float(input("  Enter Soil Moisture (%): "))

            W_req[i][j] = CalculateRequiredWater(Plant_Type[i][j], Soil_Moisture[i][j], Temperature)
            Schedule[i][j] = random.uniform(0, W_MAX)
            Fitness[i][j] = EvaluateFitness(Schedule[i][j], W_req[i][j])

    # Worker function executed in parallel for each cell (i, j)
    def update_cell(cell):
        i, j = cell
        Neighbors = GetMooreNeighbors(i, j, M, N, Schedule, Fitness)
        Seepage = ComputeSeepage(Schedule[i][j], Neighbors)
        Best_Neighbor = ArgMaxFitness(Neighbors)

        Candidate = UpdateRule(Schedule[i][j], W_req[i][j], Seepage, Best_Neighbor)
        Candidate = Clamp(Candidate, 0, W_MAX)
        Candidate_Fitness = EvaluateFitness(Candidate, W_req[i][j])

        if Candidate_Fitness > Fitness[i][j]:
            return (i, j, Candidate, Candidate_Fitness)
        else:
            return (i, j, Schedule[i][j], Fitness[i][j])

    cells = [(i, j) for i in range(M) for j in range(N)]

    # FOR iter <- 1 TO Max_Iter DO
    with ThreadPoolExecutor() as executor:
        for _ in range(Max_Iter):
            Next_Schedule = [[0.0 for _ in range(N)] for _ in range(M)]
            Next_Fitness = [[0.0 for _ in range(N)] for _ in range(M)]

            # PARALLEL FOR each cell (i, j) in M x N DO
            results = executor.map(update_cell, cells)

            for i, j, next_sched, next_fit in results:
                Next_Schedule[i][j] = next_sched
                Next_Fitness[i][j] = next_fit

            Schedule = Next_Schedule
            Fitness = Next_Fitness

    # OUTPUT Schedule, TotalWaterUsed(Schedule)
    print("\n=== Target Water Required (Liters) ===")
    for row in W_req:
        print([round(val, 2) for val in row])

    print("\n=== Optimized Watering Schedule (Liters) ===")
    for row in Schedule:
        print([round(val, 2) for val in row])

    print(f"\nTotal Water Used: {TotalWaterUsed(Schedule)} Liters")

if __name__ == "__main__":
    main()


Enter number of rows (M): 3
Enter number of columns (N): 3
Enter ambient temperature (°C): 30
Enter max iterations (Max_Iter): 80

Available plant types: succulent, grass, flower, vegetable

--- Cell (0, 0) ---
  Enter Plant Type: vegetable
  Enter Soil Moisture (%): 40

--- Cell (0, 1) ---
  Enter Plant Type: flower
  Enter Soil Moisture (%): 50

--- Cell (0, 2) ---
  Enter Plant Type: grass
  Enter Soil Moisture (%): 35

--- Cell (1, 0) ---
  Enter Plant Type: succulent
  Enter Soil Moisture (%): 15

--- Cell (1, 1) ---
  Enter Plant Type: succulent
  Enter Soil Moisture (%): 20

--- Cell (1, 2) ---
  Enter Plant Type: vegetable
  Enter Soil Moisture (%): 55

--- Cell (2, 0) ---
  Enter Plant Type: flower
  Enter Soil Moisture (%): 45

--- Cell (2, 1) ---
  Enter Plant Type: grass
  Enter Soil Moisture (%): 30

--- Cell (2, 2) ---
  Enter Plant Type: flower
  Enter Soil Moisture (%): 60

=== Target Water Required (Liters) ===
[19.5, 9.5, 10.5]
[6.75, 6.0, 12.75]
[11.25, 12.0, 6.0]

=